# Thực nghiệm Chuyên sâu: Kiểm chứng Hiện tượng Overfitting khi $r = 0$ và Hạn chế của Khoảng cách Euclidean trong `KNNModel` trước Dữ liệu Phi số (Nominal / Binary)

Notebook này mở rộng thực nghiệm từ bài báo **"KNN Model-Based Approach in Classification" (Guo et al., 2003)**, sử dụng trực tiếp pipeline tiền xử lý chung từ `data_loader.py` và hai mô hình `KNNModel` (`knn_model.py`) cùng `StandardKNN` (`baseline/standard_knn.py`) để kiểm chứng hai luận điểm quan trọng:

1. **Tính Overfitting của `KNNModel` khi $r = 0$ (Error-Tolerant Degree $r = 0$)**:
   - Khi $r = 0$, quá trình mở rộng vùng lân cận hình cầu (hypersphere) từ mỗi tâm $d_i$ buộc phải dừng ngay lập tức khi chạm phải **bất kỳ điểm dữ liệu đầu tiên nào khác nhãn**.
   - Trên các tập dữ liệu có nhiễu hoặc biên phân lớp chồng lấn, ràng buộc độ tinh khiết tuyệt đối ($r=0$) khiến không gian đặc trưng bị **phân mảnh (fragmented)** thành rất nhiều hình cầu nhỏ và các đại diện đơn điểm ($\text{Num}(d_i) = 1, \text{Sim}(d_i) = 0$). Hệ quả là mô hình khớp sát tuyệt đối tập huấn luyện ($\text{Train Accuracy} \approx 100\%$) nhưng suy giảm khả năng tổng quát hóa trên tập kiểm tra (**Generalization Gap lớn**).

2. **Nhược điểm của `KNNModel` dùng khoảng cách Euclidean trước dữ liệu Nominal và Binary (so sánh với `StandardKNN`)**:
   - Theo **Table 1** trong bài báo, 4 tập (`glass`, `iris`, `wine`, `diabetes`) chỉ chứa thuộc tính số, trong khi 2 tập **`heart`** (3 Nominal, 3 Binary, 7 Numeric) và **`aust`** (4 Nominal, 4 Binary, 6 Numeric) là dữ liệu hỗn hợp.
   - Khoảng cách Euclidean giả định không gian liên tục đẳng hướng. Khi áp dụng lên biến **Nominal** (mã hóa số nguyên gây thứ tự giả, hoặc One-Hot Encoding tạo khoảng cách nhảy bậc $\sqrt{2}$) và biến **Binary** (khoảng cách rời rạc $0$ hoặc $1$), bình phương khoảng cách Euclidean bị **chi phối bởi các bước nhảy rời rạc**, sinh ra hiện tượng **đồng khoảng cách (distance ties)** và làm suy biến ý nghĩa hình học của bán kính biên $\text{Sim}(d_i)$ trong `KNNModel`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist, pdist
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# Tự động định vị thư mục Source để import chung pipeline và model
source_path = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_loader.py").exists()),
    Path.cwd().parent.parent,
)
if str(source_path) not in sys.path:
    sys.path.append(str(source_path))

from data_loader import (
    DATASETS,
    _CATEGORICAL,
    _SHAPES,
    _information_gain,
    load_dataset,
    make_preprocessor,
)
from knn_model import KNNModel
from baseline.standard_knn import StandardKNN, get_standard_knn

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
plt.rcParams["figure.dpi"] = 110
print("Source path:", source_path)
print("Datasets:", DATASETS)

In [ ]:
# Định nghĩa cấu trúc kiểu dữ liệu của 6 bộ dữ liệu theo Table 1 (Guo et al., 2003)
DATASET_SCHEMA = {
    "glass": {"nominal": (), "binary": (), "numeric": tuple(range(9))},
    "iris": {"nominal": (), "binary": (), "numeric": tuple(range(4))},
    "heart": {
        "nominal": (2, 6, 12),          # chest-pain, electrocardiographic, thal (NN=3)
        "binary": (1, 5, 8),            # sex, fasting-blood-sugar, angina (NB=3)
        "numeric": (0, 3, 4, 7, 9, 10, 11),  # NO=7
    },
    "wine": {"nominal": (), "binary": (), "numeric": tuple(range(13))},
    "diabetes": {"nominal": (), "binary": (), "numeric": tuple(range(8))},
    "aust": {
        "nominal": (3, 4, 5, 11),       # A4, A5, A6, A12 (NN=4)
        "binary": (0, 7, 8, 10),        # A1, A8, A9, A11 (NB=4)
        "numeric": (1, 2, 6, 9, 12, 13),  # A2, A3, A7, A10, A13, A14 (NO=6)
    },
}


def build_preprocessor(
    k: int | str = "all",
    use_ohe: bool = False,
    dataset_name: str | None = None,
) -> Pipeline:
    """Pipeline tiền xử lý cục bộ theo từng fold (đồng bộ với exp.ipynb và data_loader.py)."""
    if use_ohe:
        if dataset_name is None:
            raise ValueError("Cần truyền dataset_name khi use_ohe=True.")
        return make_preprocessor(dataset_name, k=k)

    steps = [
        ("impute", SimpleImputer(strategy="median")),
        ("scale", MinMaxScaler()),
    ]
    if k != "all":
        steps.append(("select", SelectKBest(score_func=_information_gain, k=k)))
    return Pipeline(steps)


def evaluate_model_detailed(
    dataset_name: str,
    clf,
    k: int | str = "all",
    use_ohe: bool = False,
    cv=CV,
    X_override: np.ndarray | None = None,
    y_override: np.ndarray | None = None,
    custom_preprocessor: Pipeline | None = None,
) -> dict:
    """Đánh giá mô hình qua 5-fold CV, thu thập cả Train/Test Accuracy, Gap và cấu trúc Representative."""
    if X_override is not None and y_override is not None:
        X, y = X_override, y_override
    else:
        X, y = load_dataset(dataset_name)

    prep = (
        custom_preprocessor
        if custom_preprocessor is not None
        else build_preprocessor(k=k, use_ohe=use_ohe, dataset_name=dataset_name)
    )
    pipe = Pipeline([("prep", prep), ("clf", clf)])

    is_knn_model = isinstance(clf, KNNModel)
    res = cross_validate(
        pipe,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
        return_estimator=is_knn_model,
    )

    train_acc = float(res["train_score"].mean() * 100)
    test_acc = float(res["test_score"].mean() * 100)
    out = {
        "train_acc": train_acc,
        "test_acc": test_acc,
        "test_std": float(res["test_score"].std() * 100),
        "gap": train_acc - test_acc,
        "avg_reps": np.nan,
        "rr_mean": 0.0,
        "singleton_pct": np.nan,
        "small_rep_pct": np.nan,
        "avg_sim": np.nan,
        "avg_num": np.nan,
    }

    if is_knn_model:
        estimators = [est.named_steps["clf"] for est in res["estimator"]]
        out["avg_reps"] = float(np.mean([e.n_representatives_ for e in estimators]))
        out["rr_mean"] = float(np.mean([e.reduction_rate_ for e in estimators]))

        singleton_rates, small_rates, mean_sims, mean_nums = [], [], [], []
        for e in estimators:
            nums = np.array([rep.num for rep in e.representatives_], dtype=float)
            sims = np.array([rep.sim for rep in e.representatives_], dtype=float)
            singleton_rates.append(np.mean(nums == 1) * 100)
            small_rates.append(np.mean(nums <= 2) * 100)
            mean_sims.append(np.mean(sims))
            mean_nums.append(np.mean(nums))

        out["singleton_pct"] = float(np.mean(singleton_rates))
        out["small_rep_pct"] = float(np.mean(small_rates))
        out["avg_sim"] = float(np.mean(mean_sims))
        out["avg_num"] = float(np.mean(mean_nums))

    return out

---
## Phần 1: Kiểm chứng Tính Overfitting của `KNNModel` khi $r = 0$ (So sánh với $r > 0$ và `StandardKNN`)

### 1.1. Phân tích Độ chênh lệch Huấn luyện - Kiểm tra (Generalization Gap) và Sự phân mảnh Đại diện
- Khi **$r = 0, N = 0$** (không cho phép điểm khác lớp lọt vào hình cầu và không cắt tỉa đại diện nhỏ): thuật toán bắt buộc phải phủ kín 100% tập huấn luyện bằng các hình cầu thuần nhất. Các điểm nằm gần biên hoặc điểm nhiễu không thể gộp vào cụm lớn nên tách thành các **đại diện đơn điểm ($\text{Num} = 1, \text{Sim} = 0$)**.
- Khi **$r = 0, N = 1$** (loại bỏ các đại diện $\text{Num} \le 1$ rồi huấn luyện lại vẫn với $r = 0$): dù đã bỏ bớt điểm cô lập ở lượt 1, lượt huấn luyện thứ 2 với $r = 0$ vẫn bị chặn đứng bởi bất kỳ điểm khác nhãn nào trên biên, khiến bán kính $\text{Sim}(d_i)$ bị co hẹp và số đại diện vẫn lớn hơn đáng kể so với $r > 0$.
- Ta so sánh trực tiếp `KNNModel` tại $r = 0$ với $r \in \{1, 2, 3\}$ và `StandardKNN` ($k = 1, 3, 5$).

In [ ]:
overfit_rows = []

model_configs = [
    ("KNNModel (r=0, N=0)", lambda: KNNModel(r=0, N=0, metric="euclidean")),
    ("KNNModel (r=0, N=1)", lambda: KNNModel(r=0, N=1, metric="euclidean")),
    ("KNNModel (r=1, N=1)", lambda: KNNModel(r=1, N=1, metric="euclidean")),
    ("KNNModel (r=2, N=1)", lambda: KNNModel(r=2, N=1, metric="euclidean")),
    ("KNNModel (r=3, N=1)", lambda: KNNModel(r=3, N=1, metric="euclidean")),
    ("StandardKNN (k=1)", lambda: get_standard_knn(k=1)),
    ("StandardKNN (k=3)", lambda: get_standard_knn(k=3)),
    ("StandardKNN (k=5)", lambda: get_standard_knn(k=5)),
]

for ds in DATASETS:
    for model_label, model_fn in model_configs:
        m = evaluate_model_detailed(ds, model_fn())
        overfit_rows.append({
            "Dataset": ds.capitalize(),
            "Model": model_label,
            "Train Acc (%)": m["train_acc"],
            "Test Acc (%)": m["test_acc"],
            "Overfit Gap (%)": m["gap"],
            "Avg Reps": m["avg_reps"],
            "Reduction Rate (%)": m["rr_mean"],
            "Small Reps Num<=2 (%)": m["small_rep_pct"],
            "Avg Sim (Radius)": m["avg_sim"],
        })

df_overfit = pd.DataFrame(overfit_rows)

# Bảng 1A: So sánh Test Accuracy và Overfitting Gap (Train - Test) trung bình trên 6 bộ dữ liệu
summary_overfit = (
    df_overfit.groupby("Model", sort=False)[
        [
            "Train Acc (%)",
            "Test Acc (%)",
            "Overfit Gap (%)",
            "Avg Reps",
            "Reduction Rate (%)",
            "Small Reps Num<=2 (%)",
            "Avg Sim (Radius)",
        ]
    ]
    .mean()
    .round(2)
)
print("=== BẢNG 1A: TỔNG HỢP OVERFITTING GAP & CẤU TRÚC ĐẠI DIỆN TRUNG BÌNH TRÊN 6 DATASETS ===")
display(summary_overfit)

# Bảng 1B: Chi tiết Test Accuracy (%) và Overfit Gap (%) theo từng Dataset
pivot_test = df_overfit.pivot(index="Dataset", columns="Model", values="Test Acc (%)")[[
    c[0] for c in model_configs
]]
pivot_test.loc["Average"] = pivot_test.mean()
print("=== BẢNG 1B: TEST ACCURACY (%) THEO TỪNG DATASET ===")
display(pivot_test.round(2))

pivot_gap = df_overfit.pivot(index="Dataset", columns="Model", values="Overfit Gap (%)")[[
    c[0] for c in model_configs
]]
pivot_gap.loc["Average"] = pivot_gap.mean()
print("=== BẢNG 1C: OVERFITTING GAP (TRAIN ACC - TEST ACC, %) THEO TỪNG DATASET ===")
display(pivot_gap.round(2))

In [ ]:
# Vẽ đường cong Train vs Test Accuracy và Số lượng đại diện theo r (0 -> 10) trên các tập nhiều chồng lấn
target_datasets = ["diabetes", "heart", "aust", "glass"]
r_values = list(range(0, 11))

fig, axes = plt.subplots(2, 4, figsize=(18, 7.8), sharex=True)

for col_idx, ds in enumerate(target_datasets):
    train_accs_n0, test_accs_n0, reps_n0 = [], [], []
    train_accs_n1, test_accs_n1, reps_n1 = [], [], []

    knn1_test = evaluate_model_detailed(ds, get_standard_knn(k=1))["test_acc"]
    knn5_test = evaluate_model_detailed(ds, get_standard_knn(k=5))["test_acc"]

    for r_val in r_values:
        m_n0 = evaluate_model_detailed(ds, KNNModel(r=r_val, N=0, metric="euclidean"))
        m_n1 = evaluate_model_detailed(ds, KNNModel(r=r_val, N=1, metric="euclidean"))

        train_accs_n0.append(m_n0["train_acc"])
        test_accs_n0.append(m_n0["test_acc"])
        reps_n0.append(m_n0["avg_reps"])

        train_accs_n1.append(m_n1["train_acc"])
        test_accs_n1.append(m_n1["test_acc"])
        reps_n1.append(m_n1["avg_reps"])

    # Hàng 1: Train vs Test Accuracy
    ax_top = axes[0, col_idx]
    ax_top.plot(r_values, train_accs_n0, "r--o", markersize=4, alpha=0.75, label="Train (N=0)")
    ax_top.plot(r_values, test_accs_n0, "r-o", markersize=4.5, linewidth=1.8, label="Test (N=0)")
    ax_top.plot(r_values, train_accs_n1, "b--s", markersize=4, alpha=0.75, label="Train (N=1)")
    ax_top.plot(r_values, test_accs_n1, "b-s", markersize=4.5, linewidth=1.8, label="Test (N=1)")
    ax_top.axhline(knn5_test, color="darkgreen", linestyle=":", linewidth=1.5, label="StandardKNN (k=5)")
    ax_top.axvline(0, color="gray", linestyle="-.", alpha=0.5)
    ax_top.set_title(f"{ds.capitalize()}: Train vs Test Acc", fontweight="bold")
    ax_top.grid(True, alpha=0.3)
    if col_idx == 0:
        ax_top.set_ylabel("Accuracy (%)")
        ax_top.legend(fontsize=8, loc="lower left")

    # Hàng 2: Số lượng đại diện trung bình (Fragments) khi tăng r
    ax_bot = axes[1, col_idx]
    ax_bot.plot(r_values, reps_n0, "r-o", markersize=4.5, label="Avg Reps (N=0)")
    ax_bot.plot(r_values, reps_n1, "b-s", markersize=4.5, label="Avg Reps (N=1)")
    ax_bot.set_title(f"{ds.capitalize()}: Số lượng đại diện (Fragments)")
    ax_bot.set_xlabel("Error Tolerant Degree $r$")
    ax_bot.set_xticks(r_values)
    ax_bot.grid(True, alpha=0.3)
    if col_idx == 0:
        ax_bot.set_ylabel("Số đại diện trung bình")
        ax_bot.legend(fontsize=8)

plt.suptitle(
    "Hiện tượng Overfitting tại r = 0: Train Accuracy cao cực đại nhưng phân mảnh nhiều đại diện & Test Accuracy thấp",
    fontsize=13,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

### 1.2. Thí nghiệm Tiêm Nhiễu Nhãn (Label Noise Stress Test): Chứng minh $r = 0$ học vẹt nhiễu
Để kiểm chứng trực diện giả thuyết **`KNNModel(r=0)` bị overfitting vào các điểm nhiễu**, ta tiến hành tiêm nhiễu ngẫu nhiên vào nhãn của tập huấn luyện (đảo ngẫu nhiên $p\% \in \{0\%, 5\%, 10\%, 15\%, 20\%\}$ nhãn trong tập Train của mỗi fold, trong khi **giữ nguyên tập Test sạch 100%**).
- Nếu một mô hình bị overfitting khi $r=0$, mỗi điểm nhiễu trong tập Train sẽ chặn đứng quá trình mở rộng hình cầu của các điểm cùng lớp xung quanh, buộc mô hình sinh thêm hàng loạt hình cầu nhỏ bọc quanh nhiễu và làm Test Accuracy tụt dốc nhanh chóng.
- Ngược lại, khi cho phép dung nạp lỗi $r > 0$ (hoặc dùng `StandardKNN` với $k = 5$), các điểm nhiễu nằm lọt bên trong vùng lân cận đa số sẽ được bỏ qua.

In [ ]:
def evaluate_under_label_noise(
    dataset_name: str,
    noise_rates=(0.0, 0.05, 0.10, 0.15, 0.20),
    seed: int = 42,
) -> pd.DataFrame:
    """Tiêm nhiễu nhãn vào tập Train trong từng fold CV, đánh giá trên tập Test sạch."""
    X, y = load_dataset(dataset_name)
    classes = np.unique(y)
    rows = []

    models = {
        "KNNModel (r=0, N=0)": lambda: KNNModel(r=0, N=0, metric="euclidean"),
        "KNNModel (r=0, N=1)": lambda: KNNModel(r=0, N=1, metric="euclidean"),
        "KNNModel (r=2, N=1)": lambda: KNNModel(r=2, N=1, metric="euclidean"),
        "KNNModel (r=4, N=1)": lambda: KNNModel(r=4, N=1, metric="euclidean"),
        "StandardKNN (k=1)": lambda: get_standard_knn(k=1),
        "StandardKNN (k=5)": lambda: get_standard_knn(k=5),
    }

    for noise_p in noise_rates:
        for m_name, m_fn in models.items():
            fold_accs = []
            fold_reps = []

            for fold_idx, (train_idx, test_idx) in enumerate(CV.split(X, y)):
                rng = np.random.default_rng(seed + fold_idx * 100 + int(noise_p * 1000))
                X_train, y_train = X[train_idx].copy(), y[train_idx].copy()
                X_test, y_test = X[test_idx], y[test_idx]

                n_flip = int(round(noise_p * len(y_train)))
                if n_flip > 0:
                    flip_indices = rng.choice(len(y_train), size=n_flip, replace=False)
                    for idx in flip_indices:
                        other_classes = classes[classes != y_train[idx]]
                        y_train[idx] = rng.choice(other_classes)

                prep = build_preprocessor(k="all", use_ohe=False, dataset_name=dataset_name)
                X_train_trans = prep.fit_transform(X_train, y_train)
                X_test_trans = prep.transform(X_test)

                clf = m_fn()
                clf.fit(X_train_trans, y_train)
                preds = clf.predict(X_test_trans)
                fold_accs.append(accuracy_score(y_test, preds) * 100)
                if isinstance(clf, KNNModel):
                    fold_reps.append(clf.n_representatives_)

            rows.append({
                "Dataset": dataset_name.capitalize(),
                "Noise (%)": int(noise_p * 100),
                "Model": m_name,
                "Clean Test Acc (%)": float(np.mean(fold_accs)),
                "Avg Reps": float(np.mean(fold_reps)) if fold_reps else np.nan,
            })

    return pd.DataFrame(rows)


noise_datasets = ["iris", "wine", "diabetes", "aust"]
df_noise_all = pd.concat([evaluate_under_label_noise(ds) for ds in noise_datasets], ignore_index=True)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))
styles = {
    "KNNModel (r=0, N=0)": ("#d62728", "-o", 2.0),
    "KNNModel (r=0, N=1)": ("#ff7f0e", "--s", 1.8),
    "KNNModel (r=2, N=1)": ("#1f77b4", "-^", 2.0),
    "KNNModel (r=4, N=1)": ("#9467bd", "-D", 1.8),
    "StandardKNN (k=1)": ("#7f7f7f", ":x", 1.6),
    "StandardKNN (k=5)": ("#2ca02c", "-.v", 1.8),
}

for idx, ds in enumerate(noise_datasets):
    ax = axes[idx]
    sub = df_noise_all[df_noise_all["Dataset"] == ds.capitalize()]
    for m_name, (color, fmt, lw) in styles.items():
        m_sub = sub[sub["Model"] == m_name]
        ax.plot(m_sub["Noise (%)"], m_sub["Clean Test Acc (%)"], fmt, color=color, linewidth=lw, label=m_name)
    ax.set_title(f"{ds.capitalize()}: Độ bền trước nhiễu nhãn", fontweight="bold")
    ax.set_xlabel("Tỷ lệ nhiễu nhãn tập Train (%)")
    ax.set_xticks([0, 5, 10, 15, 20])
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.set_ylabel("Clean Test Accuracy (%)")
        ax.legend(fontsize=8)

plt.suptitle(
    "Kiểm chứng Overfitting: KNNModel (r=0) và 1-NN suy giảm độ chính xác mạnh khi tập Train có nhiễu",
    fontsize=12.5,
    fontweight="bold",
    y=1.03,
)
plt.tight_layout()
plt.show()

# Hiển thị mức suy giảm độ chính xác khi nhiễu tăng từ 0% -> 20%
df_drop = df_noise_all[df_noise_all["Noise (%)"].isin([0, 20])].pivot_table(
    index=["Dataset", "Model"], columns="Noise (%)", values=["Clean Test Acc (%)", "Avg Reps"]
)
df_drop[("Clean Test Acc (%)", "Drop (0->20%)")] = (
    df_drop[("Clean Test Acc (%)", 0)] - df_drop[("Clean Test Acc (%)", 20)]
)
display(df_drop.round(2))

### 1.3. Trực quan hóa Không gian 2D: Sự phân mảnh hình cầu Đại diện khi $r = 0$ so với $r = 3$ và `StandardKNN`
Biểu đồ dưới đây chiếu dữ liệu `Diabetes` (hoặc `Heart`) lên 2 đặc trưng tốt nhất theo Information Gain và vẽ trực tiếp:
- Các hình cầu đại diện $\langle \text{Rep}(d_i), \text{Sim}(d_i) \rangle$ được tạo bởi `KNNModel`.
- Miền phân lớp (Decision Boundary) của `KNNModel (r=0, N=0)`, `KNNModel (r=0, N=1)`, `KNNModel (r=3, N=1)` và `StandardKNN (k=5)`.

In [ ]:
def plot_2d_representatives_and_boundary(dataset_name: str = "diabetes"):
    X, y = load_dataset(dataset_name)
    prep_2d = build_preprocessor(k=2, use_ohe=False, dataset_name=dataset_name)
    X_2d = prep_2d.fit_transform(X, y)

    configs = [
        ("KNNModel (r=0, N=0) — Overfitting", KNNModel(r=0, N=0, metric="euclidean")),
        ("KNNModel (r=0, N=1) — Hậu cắt tỉa", KNNModel(r=0, N=1, metric="euclidean")),
        ("KNNModel (r=3, N=1) — Dung nạp lỗi r=3", KNNModel(r=3, N=1, metric="euclidean")),
        ("StandardKNN (k=5)", get_standard_knn(k=5)),
    ]

    xx, yy = np.meshgrid(np.linspace(-0.05, 1.05, 180), np.linspace(-0.05, 1.05, 180))
    grid = np.c_[xx.ravel(), yy.ravel()]
    colors = ["#1f77b4", "#d62728", "#2ca02c"]

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    for ax, (title, clf) in zip(axes, configs):
        clf.fit(X_2d, y)
        Z = clf.predict(grid).reshape(xx.shape)
        train_acc = accuracy_score(y, clf.predict(X_2d)) * 100

        ax.contourf(xx, yy, Z, alpha=0.22, cmap="coolwarm")
        for cls_idx in np.unique(y):
            mask = y == cls_idx
            ax.scatter(
                X_2d[mask, 0],
                X_2d[mask, 1],
                s=14,
                c=colors[int(cls_idx) % len(colors)],
                alpha=0.6,
                label=f"Class {cls_idx}",
            )

        if isinstance(clf, KNNModel):
            for rep in clf.representatives_:
                c = colors[int(rep.cls) % len(colors)]
                if rep.sim > 1e-6:
                    circle = Circle(
                        (rep.rep[0], rep.rep[1]),
                        rep.sim,
                        edgecolor=c,
                        facecolor="none",
                        linewidth=1.1,
                        alpha=0.8,
                    )
                    ax.add_patch(circle)
                else:
                    ax.plot(rep.rep[0], rep.rep[1], marker="x", color="black", markersize=5, alpha=0.75)
            subtitle = f"{title}\nReps={clf.n_representatives_} (RR={clf.reduction_rate_:.1f}%) | Train={train_acc:.1f}%"
        else:
            subtitle = f"{title}\nStored={len(X_2d)} (RR=0%) | Train={train_acc:.1f}%"

        ax.set_title(subtitle, fontsize=10, fontweight="bold")
        ax.set_xlim(-0.05, 1.05)
        ax.set_ylim(-0.05, 1.05)
        ax.set_xlabel("Top-1 IG Feature")
    axes[0].set_ylabel("Top-2 IG Feature")
    plt.suptitle(
        f"Trực quan hóa 2D trên {dataset_name.capitalize()}: Các hình cầu đại diện (vòng tròn) và đại diện đơn điểm Sim=0 (dấu X)",
        fontsize=12,
        fontweight="bold",
        y=1.03,
    )
    plt.tight_layout()
    plt.show()


plot_2d_representatives_and_boundary("diabetes")

---
## Phần 2: Kiểm chứng Nhược điểm của `KNNModel` dùng Khoảng cách Euclidean trước các Kiểu dữ liệu Phi số (Nominal & Binary) so với `StandardKNN`

### 2.1. Giải phẫu Ma trận Khoảng cách Euclidean trên `Heart` và `Aust`: Sự thống trị của biến rời rạc & Hiện tượng Đồng khoảng cách (Distance Ties)
Xét hai bộ dữ liệu hỗn hợp trong bài báo:
- **`Heart` (13 thuộc tính)**: 7 Numeric (`NO`), 3 Nominal (`NN`: cột `2, 6, 12`), 3 Binary (`NB`: cột `1, 5, 8`).
- **`Aust` (14 thuộc tính)**: 6 Numeric (`NO`), 4 Nominal (`NN`: cột `3, 4, 5, 11`), 4 Binary (`NB`: cột `0, 7, 8, 10`).

Khi dùng khoảng cách Euclidean $d(x, y) = \sqrt{\sum_{j} (x_j - y_j)^2}$:
1. **Tỷ trọng bình phương khoảng cách ($\Delta d^2$)**:
   - Hai mẫu khác nhau ở một thuộc tính **Binary** tạo ra độ lệch $(1 - 0)^2 = 1.0$.
   - Hai mẫu khác nhau ở một thuộc tính **Nominal** (nếu One-Hot Encode qua `make_preprocessor`) tạo ra độ lệch $1^2 + 1^2 = 2.0$ (hoặc nếu ép qua `MinMaxScaler` thì tạo ra khoảng cách thứ tự giả tạo).
   - Ngược lại, hai mẫu trên một thuộc tính **Numeric** đã chuẩn hóa $[0, 1]$ thường chỉ lệch nhau trung bình $|x_j - y_j| \approx 0.15 \implies (x_j - y_j)^2 \approx 0.0225$.
   - Hệ quả: **Bình phương khoảng cách Euclidean bị thống trị bởi các biến Binary và Nominal!**
2. **Hiện tượng Lượng tử hóa khoảng cách & Đồng khoảng cách (Distance Ties)**:
   - Trên không gian Nominal/Binary, các điểm rơi vào các đỉnh rời rạc của siêu lập phương. Rất nhiều cặp điểm có khoảng cách Euclidean bằng nhau tuyệt đối.
   - Trong `KNNModel._build_model`, bước 1 sắp xếp lân cận bằng `np.argsort(dist_matrix, axis=1)`. Khi xảy ra đồng khoảng cách giữa các điểm cùng lớp và khác lớp trên cùng một mặt cầu rời rạc, hình cầu đại diện bị dừng sớm hoặc chọn biên $\text{Sim}(d_i)$ thiếu chính xác.

In [ ]:
def analyze_euclidean_distortion(dataset_name: str) -> dict:
    """Phân tích đóng góp bình phương khoảng cách Euclidean và tỷ lệ đồng khoảng cách theo kiểu dữ liệu."""
    X, y = load_dataset(dataset_name)
    schema = DATASET_SCHEMA[dataset_name]
    num_cols = list(schema["numeric"])
    nom_cols = list(schema["nominal"])
    bin_cols = list(schema["binary"])

    # 1. Chuẩn hóa MinMax toàn bộ (use_ohe=False)
    X_raw_scaled = build_preprocessor(use_ohe=False, dataset_name=dataset_name).fit_transform(X, y)
    sq_num_raw = pdist(X_raw_scaled[:, num_cols], metric="sqeuclidean") if num_cols else np.zeros(1)
    sq_nom_raw = pdist(X_raw_scaled[:, nom_cols], metric="sqeuclidean") if nom_cols else np.zeros(1)
    sq_bin_raw = pdist(X_raw_scaled[:, bin_cols], metric="sqeuclidean") if bin_cols else np.zeros(1)
    total_sq_raw = sq_num_raw.sum() + sq_nom_raw.sum() + sq_bin_raw.sum()

    # 2. Chuẩn hóa với One-Hot Encoding cho Nominal (make_preprocessor / use_ohe=True)
    X_ohe = build_preprocessor(use_ohe=True, dataset_name=dataset_name).fit_transform(X, y)
    # Trong make_preprocessor, các cột đầu là numerical (bao gồm numeric + binary), các cột sau là OHE của nominal
    n_num_plus_bin = len(num_cols) + len(bin_cols)
    non_nom_indices = [i for i in range(_SHAPES[dataset_name][1]) if i not in nom_cols]
    num_pos_in_ohe = [non_nom_indices.index(c) for c in num_cols]
    bin_pos_in_ohe = [non_nom_indices.index(c) for c in bin_cols]
    nom_pos_in_ohe = list(range(n_num_plus_bin, X_ohe.shape[1]))

    sq_num_ohe = pdist(X_ohe[:, num_pos_in_ohe], metric="sqeuclidean") if num_pos_in_ohe else np.zeros(1)
    sq_bin_ohe = pdist(X_ohe[:, bin_pos_in_ohe], metric="sqeuclidean") if bin_pos_in_ohe else np.zeros(1)
    sq_nom_ohe = pdist(X_ohe[:, nom_pos_in_ohe], metric="sqeuclidean") if nom_pos_in_ohe else np.zeros(1)
    total_sq_ohe = sq_num_ohe.sum() + sq_bin_ohe.sum() + sq_nom_ohe.sum()

    # 3. Đếm số mức khoảng cách phân biệt & tỷ lệ điểm có khoảng cách trùng lặp (Distance Ties)
    def calc_tie_stats(X_sub):
        D = np.round(cdist(X_sub, X_sub, metric="euclidean"), 8)
        n = len(D)
        tie_Row_rates = []
        for i in range(n):
            row_d = np.delete(D[i], i)
            _, counts = np.unique(row_d, return_counts=True)
            tie_Row_rates.append(np.sum(counts[counts > 1]) / (n - 1) * 100)
        unique_dists = len(np.unique(np.round(pdist(X_sub, metric="euclidean"), 6)))
        return float(np.mean(tie_Row_rates)), unique_dists

    tie_num, uniq_num = calc_tie_stats(X_raw_scaled[:, num_cols])
    tie_non_num_raw, uniq_non_num_raw = calc_tie_stats(X_raw_scaled[:, nom_cols + bin_cols])
    tie_non_num_ohe, uniq_non_num_ohe = calc_tie_stats(X_ohe[:, bin_pos_in_ohe + nom_pos_in_ohe])

    return {
        "Dataset": dataset_name.capitalize(),
        "Numeric Cols (NO)": len(num_cols),
        "Nominal+Binary Cols (NN+NB)": len(nom_cols) + len(bin_cols),
        "[Raw MinMax] % d^2 từ Numeric": sq_num_raw.sum() / total_sq_raw * 100,
        "[Raw MinMax] % d^2 từ Nominal+Binary": (sq_nom_raw.sum() + sq_bin_raw.sum()) / total_sq_raw * 100,
        "[OHE] % d^2 từ Numeric": sq_num_ohe.sum() / total_sq_ohe * 100,
        "[OHE] % d^2 từ Nominal+Binary": (sq_nom_ohe.sum() + sq_bin_ohe.sum()) / total_sq_ohe * 100,
        "Unique Dists (Numeric Only)": uniq_num,
        "Unique Dists (Nominal+Binary Raw)": uniq_non_num_raw,
        "Unique Dists (Nominal+Binary OHE)": uniq_non_num_ohe,
        "Neighbor Tie Rate % (Numeric Only)": tie_num,
        "Neighbor Tie Rate % (Nominal+Binary OHE)": tie_non_num_ohe,
    }


df_distortion = pd.DataFrame([analyze_euclidean_distortion(ds) for ds in ("heart", "aust")]).set_index("Dataset")
print("=== BẢNG 2A: PHÂN TÍCH ĐÓNG GÓP KHOẢNG CÁCH EUCLIDEAN & TỶ LỆ ĐỒNG KHOẢNG CÁCH (TIES) ===")
display(df_distortion.T.round(2))

### 2.2. Thực nghiệm So sánh `KNNModel` và `StandardKNN` trên Từng Nhóm Kiểu Dữ liệu (`Numeric` vs. `Binary` vs. `Nominal` vs. `Mixed`)
Để thấy rõ sự suy yếu của `KNNModel` dùng Euclidean trên dữ liệu Nominal và Binary, ta tách riêng từng nhóm đặc trưng của **`Heart`** và **`Aust`** và so sánh `KNNModel` ($r=0, r=1$) với `StandardKNN` ($k=1, 3, 5$):
1. **Only Numeric (`NO`)**: Chỉ dùng các thuộc tính số liên tục/thứ tự.
2. **Only Binary (`NB`)**: Chỉ dùng các thuộc tính nhị phân $\{0, 1\}$.
3. **Only Nominal (`NN` - Raw MinMax)**: Chỉ dùng các thuộc tính phân loại (mã hóa số nguyên chuẩn hóa $[0, 1]$).
4. **Only Nominal (`NN` - One-Hot Encoded)**: Chỉ dùng các thuộc tính phân loại đã mã hóa One-Hot.
5. **Only Non-Numeric (`NN + NB` - Raw MinMax & OHE)**: Kết hợp toàn bộ các thuộc tính phi số.
6. **All Features (`NO + NN + NB` - Raw MinMax & OHE qua `make_preprocessor`)**: Toàn bộ tập đặc trưng.

In [ ]:
def make_subset_preprocessor(cols_numeric=(), cols_nominal_ohe=()):
    """Tạo ColumnTransformer cho một tập con các cột Numeric/Binary và/hoặc Nominal (OHE)."""
    transformers = []
    if cols_numeric:
        transformers.append((
            "num",
            Pipeline([
                ("impute", SimpleImputer(strategy="median")),
                ("scale", MinMaxScaler()),
            ]),
            list(cols_numeric),
        ))
    if cols_nominal_ohe:
        transformers.append((
            "nom",
            Pipeline([
                ("impute", SimpleImputer(strategy="most_frequent")),
                ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            list(cols_nominal_ohe),
        ))
    return Pipeline([("columns", ColumnTransformer(transformers))])


datatype_rows = []

for ds in ("heart", "aust"):
    X, y = load_dataset(ds)
    sc = DATASET_SCHEMA[ds]
    num_cols = sc["numeric"]
    bin_cols = sc["binary"]
    nom_cols = sc["nominal"]

    subsets = [
        ("1. Numeric Only (NO)", make_subset_preprocessor(cols_numeric=num_cols)),
        ("2. Binary Only (NB)", make_subset_preprocessor(cols_numeric=bin_cols)),
        ("3. Nominal Only (NN - Raw MinMax)", make_subset_preprocessor(cols_numeric=nom_cols)),
        ("4. Nominal Only (NN - One-Hot)", make_subset_preprocessor(cols_nominal_ohe=nom_cols)),
        ("5. Non-Numeric (NN+NB - Raw MinMax)", make_subset_preprocessor(cols_numeric=nom_cols + bin_cols)),
        ("6. Non-Numeric (NN+NB - One-Hot)", make_subset_preprocessor(cols_numeric=bin_cols, cols_nominal_ohe=nom_cols)),
        ("7. All Features (Raw MinMax)", build_preprocessor(use_ohe=False, dataset_name=ds)),
        ("8. All Features (OHE - make_preprocessor)", build_preprocessor(use_ohe=True, dataset_name=ds)),
    ]

    for subset_name, prep in subsets:
        m_r0_n0 = evaluate_model_detailed(ds, KNNModel(r=0, N=0, metric="euclidean"), custom_preprocessor=prep)
        m_r0_n1 = evaluate_model_detailed(ds, KNNModel(r=0, N=1, metric="euclidean"), custom_preprocessor=prep)
        m_r1_n1 = evaluate_model_detailed(ds, KNNModel(r=1, N=1, metric="euclidean"), custom_preprocessor=prep)

        knn1 = evaluate_model_detailed(ds, get_standard_knn(k=1), custom_preprocessor=prep)["test_acc"]
        knn3 = evaluate_model_detailed(ds, get_standard_knn(k=3), custom_preprocessor=prep)["test_acc"]
        knn5 = evaluate_model_detailed(ds, get_standard_knn(k=5), custom_preprocessor=prep)["test_acc"]
        best_std_knn = max(knn1, knn3, knn5)

        datatype_rows.append({
            "Dataset": ds.capitalize(),
            "Feature Group": subset_name,
            "KNNModel(r=0,N=0) Acc%": m_r0_n0["test_acc"],
            "KNNModel(r=0,N=1) Acc%": m_r0_n1["test_acc"],
            "KNNModel(r=1,N=1) Acc%": m_r1_n1["test_acc"],
            "StandardKNN(k=1) Acc%": knn1,
            "StandardKNN(k=3) Acc%": knn3,
            "StandardKNN(k=5) Acc%": knn5,
            "Diff (KNNModel r=0,N=1 vs k=5)": m_r0_n1["test_acc"] - knn5,
            "Diff (KNNModel r=1,N=1 vs Best kNN)": m_r1_n1["test_acc"] - best_std_knn,
            "KNNModel(r=1) Avg Sim": m_r1_n1["avg_sim"],
            "KNNModel(r=1) RR%": m_r1_n1["rr_mean"],
        })

df_datatype_ablation = pd.DataFrame(datatype_rows)
for ds in ("Heart", "Aust"):
    print(f"=== BẢNG 2B: HIỆU NĂNG TRÊN TỪNG NHÓM KIỂU DỮ LIỆU CỦA {ds.upper()} (EUCLIDEAN) ===")
    display(df_datatype_ablation[df_datatype_ablation["Dataset"] == ds].set_index("Feature Group").round(2))

In [ ]:
# Biểu đồ trực quan so sánh KNNModel (r=0, r=1) với StandardKNN (k=5) theo từng nhóm kiểu dữ liệu
fig, axes = plt.subplots(1, 2, figsize=(16, 5.2), sharey=False)

for idx, ds in enumerate(["Heart", "Aust"]):
    ax = axes[idx]
    sub = df_datatype_ablation[df_datatype_ablation["Dataset"] == ds]
    x = np.arange(len(sub))
    width = 0.25

    ax.bar(x - width, sub["KNNModel(r=0,N=1) Acc%"], width, label="KNNModel (r=0, N=1)", color="#d62728", alpha=0.85)
    ax.bar(x, sub["KNNModel(r=1,N=1) Acc%"], width, label="KNNModel (r=1, N=1)", color="#1f77b4", alpha=0.85)
    ax.bar(x + width, sub["StandardKNN(k=5) Acc%"], width, label="StandardKNN (k=5)", color="#2ca02c", alpha=0.85)

    ax.set_title(f"{ds}: KNNModel vs StandardKNN theo Kiểu dữ liệu (Euclidean)", fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(sub["Feature Group"], rotation=28, ha="right", fontsize=8.5)
    ax.set_ylabel("5-Fold CV Accuracy (%)")
    ax.set_ylim(50, 92)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

### 2.3. So sánh Khoảng cách Euclidean ($L_2$) với các Độ đo Phù hợp hơn cho Dữ liệu Phi số (Manhattan / Cityblock $L_1$, Hamming, Cosine)
Trong bài báo (Mục 3.2, trang 4), tác giả lưu ý: *"the term 'similarity measure' could be any similarity measure such as Euclidean distance or Cosine similarity only if it is suitable for a concrete application. For simplicity, from now on, we use Euclidean distance as default similarity measure"*.

- Trên các thuộc tính **Binary** và **Nominal (One-Hot)**, khoảng cách **Manhattan (`cityblock` / $L_1$)** hoặc **Hamming (`hamming`)** đo trực tiếp số lượng thuộc tính khác nhau (chuẩn $L_1$) mà không bị phóng đại phi tuyến bởi phép bình phương $\sqrt{\sum \Delta^2}$ của Euclidean ($L_2$).
- Ta thử nghiệm thay `metric="euclidean"` bằng `metric="cityblock"`, `metric="hamming"` và `metric="cosine"` trên cả `KNNModel` và `StandardKNN` đối với `Heart` và `Aust`.

In [ ]:
metrics_to_compare = ["euclidean", "cityblock", "hamming", "cosine"]
metric_rows = []

for ds in ("heart", "aust"):
    sc = DATASET_SCHEMA[ds]
    # Cấu hình 1: Chỉ các đặc trưng phi số (Nominal + Binary)
    prep_non_num = make_subset_preprocessor(cols_numeric=sc["binary"], cols_nominal_ohe=sc["nominal"])
    # Cấu hình 2: Toàn bộ đặc trưng (make_preprocessor với OHE)
    prep_all_ohe = build_preprocessor(use_ohe=True, dataset_name=ds)
    # Cấu hình 3: Toàn bộ đặc trưng (Raw MinMax)
    prep_all_raw = build_preprocessor(use_ohe=False, dataset_name=ds)

    for space_label, prep in [
        ("Non-Numeric Only (NN+NB OHE)", prep_non_num),
        ("All Features (OHE)", prep_all_ohe),
        ("All Features (Raw MinMax)", prep_all_raw),
    ]:
        for dist_metric in metrics_to_compare:
            knnm_r0 = evaluate_model_detailed(
                ds, KNNModel(r=0, N=1, metric=dist_metric), custom_preprocessor=prep
            )
            knnm_r1 = evaluate_model_detailed(
                ds, KNNModel(r=1, N=1, metric=dist_metric), custom_preprocessor=prep
            )
            std_k5 = evaluate_model_detailed(
                ds, StandardKNN(n_neighbors=5, metric=dist_metric), custom_preprocessor=prep
            )

            metric_rows.append({
                "Dataset": ds.capitalize(),
                "Feature Space": space_label,
                "Metric": dist_metric,
                "KNNModel(r=0,N=1) Acc%": knnm_r0["test_acc"],
                "KNNModel(r=1,N=1) Acc%": knnm_r1["test_acc"],
                "KNNModel(r=1,N=1) RR%": knnm_r1["rr_mean"],
                "StandardKNN(k=5) Acc%": std_k5["test_acc"],
            })

df_metrics = pd.DataFrame(metric_rows)
print("=== BẢNG 2C: SO SÁNH EUCLIDEAN VỚI MANHATTAN (CITYBLOCK), HAMMING & COSINE TRÊN HEART VÀ AUST ===")
display(
    df_metrics.pivot_table(
        index=["Dataset", "Feature Space"],
        columns="Metric",
        values=["KNNModel(r=0,N=1) Acc%", "KNNModel(r=1,N=1) Acc%", "StandardKNN(k=5) Acc%"],
    ).round(2)
)

---
## Tổng kết Kết luận Thực nghiệm (Dùng cho Báo cáo)

1. **Về hiện tượng Overfitting của `KNNModel` khi $r = 0$**:
   - Tại $r = 0$ (đặc biệt khi $N = 0$), `KNNModel` ép mọi vùng lân cận hình cầu phải tinh khiết 100% (không chứa bất kỳ điểm khác nhãn nào). Điều này dẫn đến **Train Accuracy đạt xấp xỉ 100%** nhưng **Overfitting Gap ($\text{Train Acc} - \text{Test Acc}$) rất lớn** trên các bộ dữ liệu có biên chồng lấn như `Glass`, `Diabetes`, `Heart`, `Aust`.
   - Về mặt cấu trúc, $r = 0$ gây ra hiện tượng **phân mảnh đại diện**: số lượng hình cầu sinh ra lớn nhất, bán kính trung bình $\overline{\text{Sim}(d_i)}$ nhỏ nhất, và tỷ lệ đại diện suy biến ($\text{Num} \le 2$ hoặc $\text{Sim} = 0$) rất cao.
   - Trong thí nghiệm **Tiêm nhiễu nhãn (Label Noise Stress Test)**, `KNNModel(r=0)` và `StandardKNN(k=1)` suy giảm độ chính xác nhanh nhất khi tỷ lệ nhiễu tăng từ $0\% \to 20\%$, trong khi việc nới lỏng $r > 0$ đóng vai trò như một cơ chế **chính quy hóa hình học (geometric regularization)** giúp mô hình bỏ qua các điểm nhiễu cục bộ và ổn định tương đương `StandardKNN(k=5)`.

2. **Về nhược điểm của `KNNModel` dùng khoảng cách Euclidean trước dữ liệu Nominal & Binary (`Heart`, `Aust`)**:
   - Trên `Heart` và `Aust`, các thuộc tính Nominal và Binary tạo ra các bước nhảy khoảng cách rời rạc lớn ($1.0$ hoặc $\sqrt{2}$), **chiếm phần lớn tổng bình phương khoảng cách Euclidean** ($\Delta d^2$) và làm lu mờ sự khác biệt tinh tế của các thuộc tính Numeric liên tục.
   - Khi chạy trên không gian chỉ gồm các thuộc tính phi số (**Nominal / Binary**), ma trận khoảng cách Euclidean bị lượng tử hóa mạnh với **tỷ lệ đồng khoảng cách (Distance Ties) cao**, làm suy yếu cơ chế sắp xếp khoảng cách `argsort` và cơ chế tính khoảng cách tới biên hình cầu $d(x_t, \text{Rep}) - \text{Sim}(d_i)$ của `KNNModel`, khiến `KNNModel` (nhất là tại $r=0$) kém ổn định hơn `StandardKNN(k=5)` hoặc các độ đo phù hợp hơn như `cityblock` (Manhattan) và `hamming`.